# Loan Scoring — Interactive Walkthrough

This notebook walks through the full lifecycle of the `loan_scoring` project:
building the `CreditScorer` module, running it on a sample batch, inspecting intermediate columns, saving/loading the config, and setting up the server.

Run from `projects/loan_scoring/` (the notebook's working directory).

## Setup

Add the repo root to `sys.path` so `decider` is importable, then call `initialize_decider` to load the `loan_scoring` extension package. This registers `CreditScorer` into `GraphModule` — the same call the production servers make at startup.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys, os, json, asyncio

# Repo root is two directories up from projects/loan_scoring/
PROJECT_ROOT = os.path.abspath('../..')
sys.path.insert(0, PROJECT_ROOT)

# Extension directory: decider_extensions/ lives next to this notebook
EXTENSIONS_DIR = os.path.abspath('decider_extensions')

from decider.initialization import initialize_decider

initialize_decider(extension_path=EXTENSIONS_DIR)

print('Decider initialised')
print('PROJECT_ROOT  :', PROJECT_ROOT)
print('EXTENSIONS_DIR:', EXTENSIONS_DIR)

## Import and Inspect CreditScorer

After `initialize_decider` runs, the `loan_scoring` package is on `sys.path` and its `__init__.py` has executed, registering `CreditScorer`.

In [ ]:
from loan_scoring import CreditScorer
from decider.modules import GraphModule

print('CreditScorer type identifier:', CreditScorer._CLASS_TYPE_IDENTIFIER)
print('\nModel fields:')
for field_name, field_info in CreditScorer.model_fields.items():
    default = field_info.default
    print('  ', field_name, '=', default)

## Build the Scorer and Run on Sample Batch

The sample `BATCH` below is the same data used in `generate.py`. Three applicants with different debt, income, and credit utilisation profiles.

In [ ]:
import polars as pl

# Sample batch — same as generate.py
BATCH = pl.DataFrame({
    'applicant_id': ['A001', 'A002', 'A003'],
    'debt':         [25_000.0, 5_000.0,  80_000.0],
    'income':       [50_000.0, 60_000.0, 90_000.0],
    'credit_used':  [4_000.0,  500.0,   18_000.0],
    'credit_limit': [10_000.0, 5_000.0, 20_000.0],
})

scorer = CreditScorer(
    name='credit_scorer',
    dti_weight=200.0,
    utilization_weight=100.0,
    score_base=800.0,
)

print('Scorer type   :', scorer.type)
print('dti_weight    :', scorer.dti_weight)
print('utiliz_weight :', scorer.utilization_weight)
print('score_base    :', scorer.score_base)

In [ ]:
result = scorer({'input': BATCH})
print(result)

## Inspect Intermediate Columns

The module adds all intermediate outputs to the frame. Let's look at each one in turn to understand the scoring logic:

- `dti_ratio` = `debt / income`
- `utilization_rate` = `credit_used / credit_limit`
- `credit_score_estimate` = `score_base - dti_ratio * dti_weight - utilization_rate * utilization_weight`

In [ ]:
intermediates = result.select(['applicant_id', 'dti_ratio', 'utilization_rate', 'credit_score_estimate'])
print(intermediates)

In [ ]:
# Manual check for A001:
#   dti = 25000/50000 = 0.5
#   util = 4000/10000 = 0.4
#   score = 800 - 0.5*200 - 0.4*100 = 800 - 100 - 40 = 660
a001 = result.filter(pl.col('applicant_id') == 'A001')['credit_score_estimate'][0]
print('A001 credit_score_estimate:', a001, '  (expected: 660.0)')
assert abs(a001 - 660.0) < 1e-4, 'Assertion failed'

# A002: dti=5000/60000~0.0833, util=500/5000=0.1
#   score = 800 - 0.0833*200 - 0.1*100 = 800 - 16.67 - 10 = 773.33
a002 = result.filter(pl.col('applicant_id') == 'A002')['credit_score_estimate'][0]
print('A002 credit_score_estimate:', round(a002, 2), '  (expected: ~773.33)')
assert abs(a002 - 773.33) < 0.1, 'Assertion failed'

print('All checks passed.')

## Save Config and Load it Back

In [ ]:
import tempfile
from decider.config.file import JsonFileConfigManager

CONFIGS_DIR = os.path.abspath('configs')
print('Saving to:', CONFIGS_DIR)

config_manager = JsonFileConfigManager(basepath=CONFIGS_DIR)
versioned = asyncio.run(scorer.asave('main', config_manager))
asyncio.run(config_manager.save_version(overwrite=True))

print('Saved version:', versioned.version)

In [ ]:
# Load back from disk
fresh_manager = JsonFileConfigManager(basepath=CONFIGS_DIR)
loaded = asyncio.run(fresh_manager.get_latest())

print('Loaded version:', loaded.version)

module = GraphModule.model_validate(loaded.config['main']).root
print('Reconstructed  type          :', module.type)
print('Reconstructed  dti_weight    :', module.dti_weight)
print('Reconstructed  utiliz_weight :', module.utilization_weight)
print('Reconstructed  score_base    :', module.score_base)

# Verify output matches
result2 = module({'input': BATCH})
print(result2.select(['applicant_id', 'dti_ratio', 'utilization_rate', 'credit_score_estimate']))

In [ ]:
# Inspect the JSON on disk
version_str = str(versioned.version)
json_path = os.path.join(CONFIGS_DIR, version_str, 'main.json')

with open(json_path) as f:
    on_disk = json.load(f)

print(json.dumps(on_disk, indent=2))

## Serving Instructions

Set these environment variables, then start either server. The extension path tells the server to load `loan_scoring/` at startup so `CreditScorer` is registered in `GraphModule` before any config is parsed.

```bash
export Decider_config__type=file:json
export Decider_config__basepath=/path/to/projects/loan_scoring/configs
export Decider_api__root_module=main
export Decider_ext__extension_path=/path/to/projects/loan_scoring/decider_extensions

# Starlette (uvicorn):
uvicorn decider.serving.servers.starlette:app --host 0.0.0.0 --port 8080

# Sanic:
python -m sanic decider.serving.servers.sanic:app --host 0.0.0.0 --port 8080
```

Test request:
```bash
curl -X POST http://localhost:8080/predict \
     -H 'Content-Type: application/json' \
     -d '{"debt":[25000],"income":[50000],"credit_used":[4000],"credit_limit":[10000]}'
```

Use the absolute paths printed by `generate.py` (run `python projects/loan_scoring/generate.py` from the repo root) to get the exact values for your machine.